In [1]:
import requests
import requests.auth
import os
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from datetime import datetime
from ratelimit import limits, sleep_and_retry

## Use the Pullpush API

In [3]:
@sleep_and_retry
@limits(calls=100, period=3600)  # 100 calls per hour
def fetch_posts_pullpush(subreddit, timestamp, size=100):
  """
    Fetch posts from PullPush API
  """
  url = 'https://api.pullpush.io/reddit/search/submission/'
  params = {
    'subreddit': subreddit,
    'size': size,
    'sort': 'desc',
    'before': timestamp
  }

  response = requests.get(url, params=params)
  
  current_time = datetime.now().strftime("%H:%M:%S")

  # Handle rate limiting
  if response.status_code == 429:
    print(f"[{current_time}] Rate limit exceeded. Sleeping for 1 hour...")
    time.sleep(3600)
    return fetch_posts_pullpush(subreddit, timestamp, size)

  # return error if response is not 200
  elif response.status_code != 200:
    raise ValueError(f"Error fetching posts: {response.status_code} - {response.text}")
  
  data = response.json()
  
  # Check if data is empty
  if not data.get('data'):
    print(data)
    raise ValueError("No data found in response")

  # Bug: We were previously using max which got the most recent date...
  timestamp = min([int(post['created_utc']) for post in data['data']])

  return data['data'], timestamp


def write_posts_to_csv2(posts, timestamp, subreddit):
  """
    Write posts to CSV file
  """
  now = datetime.now().strftime("%Y%m%d_%H%M%S")
  filename = f"data/{subreddit}/pullpush_run_at_{now}_to_{timestamp}.csv"
  df = pd.DataFrame(posts)
  df.to_csv(filename, index=False)

In [4]:
timestamp = int(datetime.now().timestamp())
# subreddit = 'ProductManagement'
subreddit = 'modular'
i = 0
week_of_posts = []

while True:
  i += 1
  current_time = datetime.now().strftime("%H:%M:%S")

  # Fetch posts
  # print(f"[{current_time}] Fetching posts for page {i}...")
  try:
    data, timestamp = fetch_posts_pullpush(subreddit, timestamp)
  except Exception as e:
    print(f"[{current_time}] Error fetching posts: {e}")
    week_of_posts.extend(data)
    # Write to CSV  
    write_posts_to_csv2(week_of_posts, 'error', subreddit)
    break

  # Check if there are more posts
  if not data or len(data) == 0:
    print(f"[{current_time}] No more posts found.")
    week_of_posts.extend(data)
    write_posts_to_csv2(week_of_posts, 'no_more_posts', subreddit)
    break
  
  # Append posts to list
  week_of_posts.extend(data)

  # compile a week's worth of posts and write to CSV
  if i % 7 == 0:
    print(f"[{current_time}] Writing {len(week_of_posts)} posts to CSV for week {i // 7}...")
    write_posts_to_csv2(week_of_posts, timestamp, subreddit)
    week_of_posts = []

  # Sleep for a random time between 1 and 3 seconds
  time.sleep(random.randint(1, 3))



{'error': {'code': 503, 'message': 'The Pullpush API is currently unavailable due to scheduled maintenance. Service is expected to resume by mid-May following necessary hardware upgrades and data reindexing.', 'status': 'SERVICE_UNAVAILABLE', 'retryAfter': '2025-05-15T00:00:00Z'}}
[09:34:35] Error fetching posts: No data found in response


NameError: name 'data' is not defined

## [DEPRECATED] Scraping

In [3]:
url = "https://www.reddit.com/r/ProductManagement/"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1'
}

response = requests.get(url, headers=headers)
response.raise_for_status()  # Raise an exception for HTTP errors

In [5]:
soup = BeautifulSoup(response.text, 'html.parser')

# Find the shreddit-feed element
feed = soup.find('shreddit-feed')

In [ ]:
# List to store post data
posts_data = []

# Find all posts within the feed
posts = feed.find_all('article')

for post in posts:

  # Extract title
  title_element = post.find('a')
  title = title_element.text.strip()

  # Extract preview text
  preview_element = post.find('a', attrs={'slot': 'text-body'})
  preview = preview_element.text.strip()

  # Get remaining metadata
  shreddit_post = post.find('shreddit-post')
  comments = shreddit_post.get('comment-count')
  score = shreddit_post.get('score')
  created_at = shreddit_post.get('created-timestamp')

  # Append to list
  posts_data.append({
      'title': title,
      'preview': preview,
      'comments': comments,
      'upvotes': score,
      'created_at': created_at
  })



In [20]:
posts_data

[{'title': 'Collaboration skills are one of the most underrated skills needed for a product manager, lets discuss about them, what execeptional skills did you see in your colleagues or yourself that made life easy for everyone?',
  'preview': "I will start,\n  \n    here's a trick i learnt from my senior product manager who is a stalwart in our org\n  \n    So, my colleague always takes the time to meet with stakeholders 1:1 before sharing his features in group stakeholder sessions. By doing 1:1s, it helps him build better relationships, get early feedback, and make sure he's on the right track. Plus, when there’s some misalignment, those 1:1s give him the insights he needs to tweak or back up his proposals. So by the time the bigger stakeholder meetings happen, everything flows super smooth like butter and he always ends up getting immense praises from our VP of Product.\n  \n    If possible, can you all share specific examples? that way our discussion can be more engaging",
  'commen

In [ ]:
# Convert to df
df = pd.DataFrame(posts_data)
        
# Export to CSV
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"reddit_product_management_{timestamp}.csv"
df.to_csv(filename, index=False)

## [DEPRECATED] Use the API

In [6]:
# Your Reddit app credentials
client_id = os.getenv('REDDIT_CLIENT_ID')
client_secret = os.getenv('REDDIT_CLIENT_SECRET')
username = os.getenv('REDDIT_USERNAME')
password = os.getenv('REDDIT_PASSWORD')

# Check if all credentials are set
if not all([client_id, client_secret, username, password]):
    raise ValueError("Please set the Reddit app credentials in environment variables.")

# Create authentication object
auth = requests.auth.HTTPBasicAuth(client_id, client_secret)

# Request data
data = {
    'grant_type': 'password',
    'username': username,
    'password': password
}



# User agent is required by Reddit's API rules
headers = {'User-Agent': f'PmSentimentScraper/0.1 (by /u/{username})'}

# Make the POST request to get the token
response = requests.post(
    'https://www.reddit.com/api/v1/access_token',
    auth=auth,
    data=data,
    headers=headers
)

# Extract the token
token_data = response.json()

access_token = token_data['access_token']
# print(f"Access token: {access_token}")




In [22]:

def fetch_posts(subreddit, access_token, after=None, posts_per_page=100):
  """
    Fetch Reddit posts using OAuth
  """
  all_posts = []

  headers = {
    'Authorization': f'bearer {access_token}',
    'User-Agent': f'PmSentimentScraper/0.1 (by /u/{os.getenv("REDDIT_USERNAME")})'
  }

  params = {'limit': posts_per_page}
  if after:
    params['after'] = after

  response = requests.get(
      f'https://oauth.reddit.com/r/{subreddit}', 
      headers=headers, 
      params=params
  )

  # Print the current ratelimit status
  # https://support.reddithelp.com/hc/en-us/articles/16160319875092-Reddit-Data-API-Wiki
  print(f"requests used: {response.headers['x-ratelimit-used']}")
  print(f"requests remaining: {response.headers['x-ratelimit-remaining']}")
  print(f"secs until ratelimit reset: {response.headers['x-ratelimit-reset']}")

  data = response.json()

  for post in data['data']['children']:
    post_data = post['data']
    all_posts.append(post_data)

  after = data['data']['after']

  return all_posts, after


In [ ]:
subreddit = 'ProductManagement'
headers = {
  'Authorization': f'bearer {access_token}',
  'User-Agent': f'PmSentimentScraper/0.1 (by /u/{os.getenv("REDDIT_USERNAME")})'
}

response = requests.get(
    f'https://oauth.reddit.com/r/{subreddit}', 
    headers=headers,
)

data = response.json()
print(f'requests used: {response.headers['x-ratelimit-used']}')
print(f'requests remaining: {response.headers['x-ratelimit-remaining']}')
print(f'secs until ratelimit reset: {response.headers['x-ratelimit-reset']}')

{'Connection': 'keep-alive', 'Content-Length': '15725', 'x-ua-compatible': 'IE=edge', 'content-type': 'application/json; charset=UTF-8', 'expires': '-1', 'cache-control': 'private, s-maxage=0, max-age=0, must-revalidate, no-store', 'content-encoding': 'gzip', 'x-ratelimit-used': '7', 'x-ratelimit-remaining': '993.0', 'x-ratelimit-reset': '62', 'Accept-Ranges': 'bytes', 'Date': 'Thu, 17 Apr 2025 15:58:58 GMT', 'Via': '1.1 varnish', 'Vary': 'accept-encoding', 'Strict-Transport-Security': 'max-age=31536000; includeSubdomains', 'X-Content-Type-Options': 'nosniff', 'X-Frame-Options': 'SAMEORIGIN', 'X-XSS-Protection': '1; mode=block', 'Set-Cookie': 'loid=0000000009gl8ai9u3.2.1681784192000.Z0FBQUFBQm9BU1ZDaTlJY2t2MzZ4MmNTZFpqUUtGeDhFRXJTVHNRTlhZNE1ETGh1c1NsR2ZadXJIeUltdUdBeHAwX2ZILWdWYkljcjBSUWFJZUFJOV9meEluQTd5YXRzZUwzdG5odnc2YVloVHpFaDRPOE12SW9BcElySktFYkdBVk5kSHRselNjSE4; Domain=reddit.com; Max-Age=63071999; Path=/; expires=Sat, 17-Apr-2027 15:58:58 GMT; secure; SameSite=None; Secure, sess

In [34]:
def write_posts_to_csv(posts, after=None):
  """
    Write posts to CSV file
  """
  timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
  if after:
    filename = f"reddit_product_management_{timestamp}_after_{after}.csv"
  else:
    filename = f"reddit_product_management_{timestamp}.csv"
  df = pd.DataFrame(posts)
  df.to_csv(filename, index=False)

In [23]:
subreddit = 'ProductManagement'
after = 't3_1in19mm'
i = 0
while True:
  print(f"Fetching posts for page {i}...")
  i += 1
  try:
    # Fetch posts
    posts_data, after = fetch_posts(subreddit, access_token, after=after)
  except requests.exceptions.RequestException as e:
    print(f"Error fetching posts: {e}")
    break

  # Write to CSV
  write_posts_to_csv(posts_data, after)

  # Check if there are more posts
  if not after:
    break

  # Sleep for a random time between 1 and 5 seconds
  time.sleep(random.randint(1, 5))

Fetching posts for page 0...
requests used: 1
requests remaining: 999.0
secs until ratelimit reset: 491


## JUNK / NOTES

In [41]:
test = []
test.extend([1, 2, 3])
test


[1, 2, 3]

In [ ]:

def fetch_posts(subreddit, access_token, start_page=1, pages=5, posts_per_page=25):
    
    all_posts = []
    after = None
    
    for i in range(pages):
        url = f"https://www.reddit.com/r/{subreddit}.json?limit={posts_per_page}"
        if after:
            url += f"&after={after}"
            
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        
        response = requests.get(url, headers=headers)
        data = response.json()
        
        # Extract posts
        for post in data['data']['children']:
            post_data = post['data']
            all_posts.append(post_data)
            # print(post_data.keys())
            # all_posts.append({
            #     'title': post_data['title'],
            #     'upvotes': post_data['score'],
            #     'comment_count': post_data['num_comments']
            #     # Add other fields as needed
            # })
        
        # Get the 'after' parameter for the next page
        after = data['data']['after']
        if not after:
            break  # No more pages
            
        # Respect rate limits
        time.sleep(3)
    
    return all_posts

In [ ]:
# Convert to df
df = pd.DataFrame(data)
        
# Export to CSV
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"reddit_product_management_{timestamp}.csv"
df.to_csv(filename, index=False)

In [28]:
df.head()

,approved_at_utc,subreddit,selftext,author_fullname,saved,mod_reason_title,gilded,clicked,title,link_flair_richtext,...,num_crossposts,media,is_video,link_flair_template_id,url_overridden_by_dest,media_metadata,poll_data,author_cakeday,crosspost_parent_list,crosspost_parent
0,None,ProductManagement,Share your frustrations and get support/feedba...,t2_6l4z3,False,None,0,False,Weekly rant thread,[],...,0,None,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,None,ProductManagement,Share your frustrations and get support/feedba...,t2_6l4z3,False,None,0,False,Weekly rant thread,[],...,0,None,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,None,ProductManagement,"I will start,\n\nhere's a trick i learnt from ...",t2_je5wkn0wd,False,None,0,False,Collaboration skills are one of the most under...,[],...,0,None,False,8acd9e18-5d3d-11eb-871d-0e9445fd5f7f,NaN,NaN,NaN,NaN,NaN,NaN
3,None,ProductManagement,A lot of the work I do as a PM these days is b...,t2_kw6fqr0a6,False,None,0,False,Drained and stressed out,[],...,0,None,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,None,ProductManagement,We all know we have a ton of meetings as PMs b...,t2_eoj3s4py,False,None,0,False,How do you limit meetings,[],...,0,None,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
